# 03. Baseline Classifier Training (TF-IDF + Logistic Regression)
### Clinical Note-to-Billing Code Consistency Checker (FWA Triage)

Trains a CPU-friendly, interpretable multi-class baseline classifier predicting the expected code category from note content, combined with text-length and entity count features.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from clinical_billing_audit.models.baseline import BaselineClassifier

## 1. Load Processed Paired Dataset

In [ ]:
data_path = Path("../data/processed/paired_with_mismatches.csv")
if not data_path.exists():
    print("Paired dataset not found. Run pipeline first or load from raw.")
    df = pd.read_csv("../data/raw/mtsamples.csv").dropna(subset=['transcription'])
    df['code_category'] = 'general'
else:
    df = pd.read_csv(data_path)

print(f"Total dataset size: {len(df):,}")

## 2. Stratified Train / Test Split

In [ ]:
texts = df['transcription'].tolist()
labels = df['code_category'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)
print(f"Train samples: {len(X_train):,}, Test samples: {len(X_test):,}")

## 3. Train Baseline Model & Cross-Validation

In [ ]:
clf = BaselineClassifier(
    tfidf_params={"max_features": 15000, "ngram_range": (1, 2)},
    lr_params={"C": 1.0, "max_iter": 1000, "class_weight": "balanced"}
)

train_res = clf.fit(X_train, y_train)
print(f"Training Weighted F1: {train_res['train_f1_weighted']:.4f}")

eval_report = clf.evaluate(X_test, y_test)
print(f"Test Accuracy: {eval_report['accuracy']:.4f}")
print(f"Test Weighted F1: {eval_report['weighted avg']['f1-score']:.4f}")

## 4. Top Feature Importances per Category

In [ ]:
importance_df = clf.get_feature_importance(top_n=10)
importance_df.head(15)